# Assignment 1A Part B
## Instruction Fine-Tuning with QLoRA

This notebook converts the cleaned Cisco and networking corpus from Part A into a grounded instruction dataset, fine-tunes three QLoRA adapters on the saved Qwen2.5-1.5B CPT checkpoint, and compares their held-out loss, efficiency and generated-answer quality.

## Table of Contents

1. [Part B1 Instruction Dataset Creation](#part-b1-instruction-dataset-creation)
2. [Part B2 QLoRA Fine-Tuning](#part-b2-qlora-fine-tuning)
   - [Runtime and GPU Setup](#runtime-and-gpu-setup)
   - [QLoRA Training Implementation](#qlora-training-implementation)
   - [Adapter A](#adapter-a)
   - [Adapter B](#adapter-b)
   - [Adapter C](#adapter-c)
3. [Part B3 Comparative Evaluation](#part-b3-comparative-evaluation)
4. [Final Results and Recommendation](#final-results-and-recommendation)

<a id="part-b1-instruction-dataset-creation"></a>
## Part B1 Instruction Dataset Creation

This stage uses the 190 cleaned text documents produced in Part A as the sole knowledge source for 200 LLM-authored `instruction` and `response` records. The responses are concise paraphrases grounded in the submitted corpus and were reviewed for relevance and factual consistency. The master dataset is deliberately balanced between 100 Cisco product and operations examples and 100 networking-protocol and OSPF/RFC examples.

The complete collection is stored in `data/lora-instructions/instruction_dataset.json`. The following cell performs only a reproducible, seeded random 80:20 split and writes 160 training records and 40 evaluation records in JSONL format. Seed 7 produces balanced splits containing 80 Cisco and 80 protocol examples for training and 20 Cisco and 20 protocol examples for evaluation. The next two cells display one record from each split to verify the required structure.

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MODEL = "Qwen/Qwen2.5-1.5B"
MODEL_KEY = "Qwen2.5-1.5B"
RUN_VERSION = "v3"

MODEL_OUTPUT_DIR = PROJECT_ROOT / "output" / MODEL_KEY
RUN_OUTPUT_DIR = MODEL_OUTPUT_DIR / RUN_VERSION
CPT_MODEL_DIR = RUN_OUTPUT_DIR / "final_model"
EVALUATION_OUTPUT_DIR = (
    PROJECT_ROOT / "evaluation" / MODEL_KEY / RUN_VERSION
)
INSTRUCTION_DATA_DIR = PROJECT_ROOT / "data" / "lora-instructions"
INSTRUCTION_DATA_FILE = INSTRUCTION_DATA_DIR / "instruction_dataset.json"

# sys.path.append(os.path.abspath('./'))
# # Instructon Generation logic is a large python code, 
# # so not including in notebook, just loading from py
# from create_instruction_dataset import create_instruction_dataset
# report = create_instruction_dataset(
#             PROJECT_ROOT / "data" / "cleaned",
#             INSTRUCTION_DATA_DIR,
#             total_pairs=200,
#             train_ratio=0.8,
#             max_pairs_per_document=10,
#             seed=43,
#         )
# train = report["train"]
# evaluation = report["evaluation"]
# print("=== Instruction Dataset Summary ===")
# print(f"Input text files:       {report['input_text_file_count']:,}")
# print(f"Usable unique pairs:    {report['candidate_pair_count_after_deduplication']:,}")
# print(f"Training pairs:         {train['count']:,}")
# print(f"Evaluation pairs:       {evaluation['count']:,}")
# print(f"Source overlap:         {report['source_overlap_count']}")
# print(f"Grounding check:        {report['grounding_check']}")

In [2]:
import argparse
import json
import random
from pathlib import Path


def split_instruction_dataset(input_file, output_dir, seed=7):
    records = json.loads(Path(input_file).read_text(encoding="utf-8"))
    random.Random(seed).shuffle(records)

    split_at = int(len(records) * 0.80)
    train_records = records[:split_at]
    evaluation_records = records[split_at:]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, rows in (("train.jsonl", train_records), ("evaluation.jsonl", evaluation_records)):
        with (output_dir / name).open("w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Seed: {seed}")
    print(f"Total examples: {len(records)}")
    print(f"Training examples: {len(train_records)}")
    print(f"Evaluation examples: {len(evaluation_records)}")


split_instruction_dataset(INSTRUCTION_DATA_FILE, INSTRUCTION_DATA_DIR)

Seed: 7
Total examples: 200
Training examples: 160
Evaluation examples: 40


In [3]:
!tail -1 "data/lora-instructions/evaluation.jsonl"

{"instruction": "What does Nexus Hyperfabric continuously validate after deployment?", "response": "It uses assertion-based monitoring to verify fabric and connected-resource availability and reliability and to identify root causes when problems are detected."}


In [4]:
!head -1 "data/lora-instructions/train.jsonl"

{"instruction": "What operational role does Cisco Nexus Dashboard provide for on-premises data centers?", "response": "Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations."}


<a id="part-b2-qlora-fine-tuning"></a>
## Part B2 QLoRA Fine-Tuning with Three Adapter Configurations

All three experiments start from the same domain-adapted Qwen2.5-1.5B CPT checkpoint and use the same 160/40 instruction split. The frozen base model is loaded in 4-bit NF4 form, while small trainable LoRA matrices are attached to selected attention projections. Keeping the data, seed and training hyperparameters fixed isolates the effect of adapter capacity.

| Adapter | Capacity | Rank | Alpha | Target modules |
|---|---|---:|---:|---|
| A | Low | 8 | 16 | `q_proj`, `v_proj` |
| B | Balanced | 16 | 32 | `q_proj`, `v_proj` |
| C | High | 32 | 32 | `q_proj`, `v_proj`, `o_proj` |

<a id="runtime-and-gpu-setup"></a>
### Runtime and GPU Setup

The following cell restricts the run to one visible CUDA device and prints the selected GPU. The submitted experiments ran on one NVIDIA A100 80 GB GPU, providing native bfloat16 support for QLoRA computation.

In [5]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch

print("Visible GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.device_count() == 1


Visible GPU count: 1
GPU: NVIDIA A100-SXM4-80GB


In [6]:
import math
import os
from pathlib import Path

import torch
from datasets import load_dataset
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

<a id="qlora-training-implementation"></a>
### QLoRA Training Implementation

The next cell defines the common training function used by all three adapters. It validates and applies the Qwen chat template, loads the CPT checkpoint with 4-bit NF4 quantization, prepares it for k-bit training and attaches the requested LoRA configuration. Instruction tokens and padding are masked with `-100`, so only assistant-response tokens contribute to the supervised loss. `SFTTrainer` evaluates the held-out set before training and after every epoch, saves checkpoints and writes the final adapter and tokenizer.

In [7]:

def run_qlora_finetuning_with_chat_template(
    # Model and data
    model_name,
    train_file,
    evaluation_file,
    output_dir,
    cache_dir=None,

    # LoRA adapter
    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules="all-linear",
    lora_bias="none",

    # QLoRA quantization
    quantization_type="nf4",
    use_double_quantization=True,

    # Training
    epochs=3,
    learning_rate=1e-4,
    train_batch_size=1,
    evaluation_batch_size=1,
    gradient_accumulation_steps=8,
    max_length=1024,
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=1.0,
    optimizer="paged_adamw_8bit",
    lr_scheduler_type="linear",

    # Logging and checkpoints
    logging_steps=5,
    disable_tqdm=True,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=2,

    # Memory and reproducibility
    gradient_checkpointing=True,
    packing=False,
    seed=42,
    trust_remote_code=False,
    resume_from_checkpoint=None,
):
    """
    Fine-tune a causal language model using 4-bit QLoRA.

    Expected JSONL dataset format:
        {"instruction": "...", "response": "..."}

    Each instruction/response pair is formatted with the tokenizer's
    model-specific chat template before tokenization.

    Only assistant-response tokens contribute to training loss.
    Instruction and padding tokens receive labels of -100.
    """

    # ---------------------------------------------------------
    # Validate runtime and arguments
    # ---------------------------------------------------------

    if not torch.cuda.is_available():
        raise RuntimeError(
            "QLoRA training requires a CUDA-capable GPU."
        )

    if torch.cuda.device_count() != 1:
        raise RuntimeError(
            f"{torch.cuda.device_count()} GPUs are visible. "
            "This notebook configuration expects exactly one visible GPU. "
            "Set CUDA_VISIBLE_DEVICES=0 before importing torch, then "
            "restart the kernel."
        )

    if packing:
        raise ValueError(
            "packing=True is not supported because this implementation "
            "constructs explicit response-only labels."
        )

    if max_length <= 0:
        raise ValueError("max_length must be greater than zero.")

    if lora_rank <= 0:
        raise ValueError("lora_rank must be greater than zero.")

    train_file = Path(train_file).expanduser().resolve()
    evaluation_file = Path(evaluation_file).expanduser().resolve()
    output_dir = Path(output_dir).expanduser().resolve()

    if not train_file.is_file():
        raise FileNotFoundError(
            f"Training dataset does not exist: {train_file}"
        )

    if not evaluation_file.is_file():
        raise FileNotFoundError(
            f"Evaluation dataset does not exist: {evaluation_file}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    if cache_dir is None:
        cache_dir = os.getenv("CACHE_DIR")

    if cache_dir:
        cache_dir = str(Path(cache_dir).expanduser().resolve())

    set_seed(seed)

    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    gpu_major_version = torch.cuda.get_device_capability(0)[0]
    use_tf32 = gpu_major_version >= 8

    print("=" * 60)
    print("QLoRA TRAINING CONFIGURATION")
    print("=" * 60)
    print(f"Model/CPT checkpoint:         {model_name}")
    print(f"Training data:                {train_file}")
    print(f"Evaluation data:              {evaluation_file}")
    print(f"Output directory:             {output_dir}")
    print(f"GPU:                          {torch.cuda.get_device_name(0)}")
    print(f"Compute dtype:                {compute_dtype}")
    print(f"TF32 enabled:                 {use_tf32}")
    print(f"LoRA rank:                    {lora_rank}")
    print(f"LoRA alpha:                   {lora_alpha}")
    print(f"LoRA scaling:                 {lora_alpha / lora_rank:.2f}")
    print(f"LoRA dropout:                 {lora_dropout}")
    print(f"Target modules:               {target_modules}")
    print(f"Epochs:                       {epochs}")
    print(f"Learning rate:                {learning_rate}")
    print(f"Micro-batch size:             {train_batch_size}")
    print(f"Gradient accumulation:        {gradient_accumulation_steps}")
    print(
        "Effective batch size:          "
        f"{train_batch_size * gradient_accumulation_steps}"
    )
    print(f"Maximum sequence length:      {max_length}")
    print(f"Progress bar disabled:        {disable_tqdm}")
    print("=" * 60)

    # ---------------------------------------------------------
    # Load tokenizer
    # ---------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        trust_remote_code=trust_remote_code,
        use_fast=True,
    )

    if tokenizer.eos_token is None:
        raise ValueError(
            "The selected tokenizer does not define an EOS token."
        )

    # This is safe with our custom collator because padding labels
    # are explicitly set to -100, while genuine EOS labels remain.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    chat_template = getattr(tokenizer, "chat_template", None)

    if not chat_template:
        raise ValueError(
            "The selected tokenizer does not define a chat template. "
            "Assignment 1A requires SFT data to use the model's chat "
            "template. Use a compatible chat/instruct tokenizer or save "
            "an explicit chat template with the CPT checkpoint before "
            "running QLoRA."
        )

    # This smoke test validates not only that chat_template is populated,
    # but also that this tokenizer can resolve and execute the template.
    try:
        tokenizer.apply_chat_template(
            [{"role": "user", "content": "Template validation"}],
            tokenize=True,
            add_generation_prompt=True,
        )
    except Exception as error:
        raise ValueError(
            "The tokenizer has a chat template, but it could not be "
            "applied to user/assistant messages."
        ) from error

    print(
        f"EOS token: {tokenizer.eos_token!r} "
        f"(ID: {tokenizer.eos_token_id})"
    )
    print(
        f"PAD token: {tokenizer.pad_token!r} "
        f"(ID: {tokenizer.pad_token_id})"
    )
    print("Chat template: detected and validated")

    # ---------------------------------------------------------
    # Configure 4-bit base-model loading
    # ---------------------------------------------------------

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=quantization_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=use_double_quantization,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=cache_dir,
        quantization_config=quantization_config,
        device_map={"": 0},
        torch_dtype=compute_dtype,
        trust_remote_code=trust_remote_code,
    )

    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id

    # Prepare the quantized model for adapter training.
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=gradient_checkpointing,
    )

    # ---------------------------------------------------------
    # Attach LoRA adapter
    # ---------------------------------------------------------

    lora_config = LoraConfig(
        task_type="CAUSAL_LM",
        inference_mode=False,
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias=lora_bias,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # ---------------------------------------------------------
    # Load JSONL datasets
    # ---------------------------------------------------------

    train_dataset = load_dataset(
        "json",
        data_files=str(train_file),
        split="train",
    )

    evaluation_dataset = load_dataset(
        "json",
        data_files=str(evaluation_file),
        split="train",
    )

    required_columns = {"instruction", "response"}

    for dataset_name, dataset in (
        ("training", train_dataset),
        ("evaluation", evaluation_dataset),
    ):
        missing_columns = (
            required_columns - set(dataset.column_names)
        )

        if missing_columns:
            raise ValueError(
                f"The {dataset_name} dataset is missing columns: "
                f"{sorted(missing_columns)}"
            )

        if len(dataset) == 0:
            raise ValueError(
                f"The {dataset_name} dataset contains no records."
            )

    print(f"Training examples:            {len(train_dataset):,}")
    print(f"Evaluation examples:          {len(evaluation_dataset):,}")

    # ---------------------------------------------------------
    # Format records
    # ---------------------------------------------------------

    def format_record(record):
        """
        Render the record with the tokenizer's model-specific chat
        template for inspection. Token IDs are created separately below.
        """
        instruction = record["instruction"].strip()
        response = record["response"].strip()

        if not instruction:
            raise ValueError("Encountered an empty instruction.")

        if not response:
            raise ValueError("Encountered an empty response.")

        prompt_messages = [
            {"role": "user", "content": instruction},
        ]
        full_messages = [
            *prompt_messages,
            {"role": "assistant", "content": response},
        ]

        return {
            "prompt_text": tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True,
            ),
            "text": tokenizer.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False,
            ),
        }

    train_dataset = train_dataset.map(format_record)
    evaluation_dataset = evaluation_dataset.map(format_record)

    print("\nFormatted training example:")
    print(train_dataset[0]["text"][:1000])

    # ---------------------------------------------------------
    # Explicit response-only tokenization
    # ---------------------------------------------------------

    def tokenize_record(record):
        """
        Construct causal-LM labels explicitly.

        Theory:
        A causal language model normally predicts every token in the
        sequence. For supervised instruction tuning, we only want the
        answer to contribute to loss.

        Implementation:
        - Prompt labels are set to -100.
        - Response labels contain their actual token IDs.
        - Chat-template assistant/end tokens remain in response labels.
        - Padding labels are later set to -100 by the collator.
        """
        instruction = record["instruction"].strip()
        response = record["response"].strip()

        prompt_messages = [
            {"role": "user", "content": instruction},
        ]
        full_messages = [
            *prompt_messages,
            {"role": "assistant", "content": response},
        ]

        # add_generation_prompt=True includes the model-specific assistant
        # prefix. It is context, not an answer, so its labels are masked.
        prompt_ids = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
        )
        full_ids = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=False,
        )

        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError(
                "The tokenizer's completed chat is not prefixed by its "
                "generation prompt, so the assistant-response boundary "
                "cannot be masked safely."
            )

        if len(prompt_ids) >= max_length:
            raise ValueError(
                "The instruction leaves no room for a response. "
                f"Prompt tokens: {len(prompt_ids)}, "
                f"max_length: {max_length}"
            )

        # Everything after the generation prompt is the assistant target,
        # including model-specific end-of-message control tokens.
        response_ids = full_ids[len(prompt_ids):]
        response_ids = response_ids[:max_length - len(prompt_ids)]

        if not response_ids:
            raise ValueError(
                "The chat template produced no assistant-response tokens."
            )

        input_ids = prompt_ids + response_ids

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": (
                [-100] * len(prompt_ids)
                + response_ids.copy()
            ),
        }

    tokenized_train_dataset = train_dataset.map(
        tokenize_record,
        remove_columns=train_dataset.column_names,
    )

    tokenized_evaluation_dataset = evaluation_dataset.map(
        tokenize_record,
        remove_columns=evaluation_dataset.column_names,
    )

    # ---------------------------------------------------------
    # Custom padding collator
    # ---------------------------------------------------------

    def completion_collator(features):
        """
        Pad each batch dynamically.

        Input IDs use the tokenizer's pad token.
        Attention is zero on padding.
        Labels use -100 on padding so it is ignored by loss.
        """
        batch_max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_masks = []
        batch_labels = []

        for feature in features:
            padding_length = (
                batch_max_length - len(feature["input_ids"])
            )

            batch_input_ids.append(
                feature["input_ids"]
                + [tokenizer.pad_token_id] * padding_length
            )

            batch_attention_masks.append(
                feature["attention_mask"]
                + [0] * padding_length
            )

            batch_labels.append(
                feature["labels"]
                + [-100] * padding_length
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                batch_attention_masks,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                batch_labels,
                dtype=torch.long,
            ),
        }

    # ---------------------------------------------------------
    # Verify response-only labels
    # ---------------------------------------------------------

    sample = tokenized_train_dataset[0]

    trained_token_ids = [
        token_id
        for token_id in sample["labels"]
        if token_id != -100
    ]

    assert trained_token_ids, (
        "No response tokens contribute to training loss."
    )

    print(
        "\nTokens contributing to loss:",
        len(trained_token_ids),
    )

    print(
        "Text contributing to loss:",
        tokenizer.decode(
            trained_token_ids,
            skip_special_tokens=False,
        )[:1000],
    )

    test_batch = completion_collator(
        [
            tokenized_train_dataset[0],
            tokenized_train_dataset[
                min(1, len(tokenized_train_dataset) - 1)
            ],
        ]
    )

    assert (
        test_batch["input_ids"].shape
        == test_batch["attention_mask"].shape
        == test_batch["labels"].shape
    ), "The collated input, mask, and label shapes do not match."

    print(
        "Validated batch shape:",
        tuple(test_batch["input_ids"].shape),
    )

    # ---------------------------------------------------------
    # Trainer configuration for TRL 0.12.1
    # ---------------------------------------------------------

    use_bf16 = compute_dtype == torch.bfloat16
    use_fp16 = compute_dtype == torch.float16

    training_config = SFTConfig(
        output_dir=str(output_dir),

        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=evaluation_batch_size,
        gradient_accumulation_steps=(
            gradient_accumulation_steps
        ),

        max_seq_length=max_length,

        # Data is already tokenized and labels already exist.
        packing=False,
        remove_unused_columns=False,
        dataset_kwargs={
            "skip_prepare_dataset": True,
        },

        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        max_grad_norm=max_grad_norm,
        lr_scheduler_type=lr_scheduler_type,
        optim=optimizer,

        bf16=use_bf16,
        fp16=use_fp16,
        tf32=use_tf32,

        gradient_checkpointing=gradient_checkpointing,
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        },

        eval_strategy=evaluation_strategy,
        save_strategy=save_strategy,
        save_total_limit=save_total_limit,

        logging_strategy="steps",
        logging_steps=logging_steps,
        logging_first_step=True,
        disable_tqdm=disable_tqdm,

        dataloader_num_workers=0,

        seed=seed,
        data_seed=seed,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_config,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_evaluation_dataset,
        data_collator=completion_collator,
        processing_class=tokenizer,
    )

    # ---------------------------------------------------------
    # Evaluate before training
    # ---------------------------------------------------------

    print("\nEvaluating before adapter training...")

    initial_evaluation = trainer.evaluate(
        metric_key_prefix="before_training"
    )

    initial_loss = initial_evaluation.get(
        "before_training_loss"
    )

    initial_perplexity = (
        math.exp(initial_loss)
        if initial_loss is not None and initial_loss < 100
        else float("inf")
    )

    print(f"Initial evaluation loss:       {initial_loss}")
    print(f"Initial evaluation perplexity: {initial_perplexity}")

    trainer.save_metrics(
        "before_training",
        initial_evaluation,
    )

    # ---------------------------------------------------------
    # Train
    # ---------------------------------------------------------

    print("\nStarting QLoRA training...")

    training_result = trainer.train(
        resume_from_checkpoint=resume_from_checkpoint
    )

    trainer.log_metrics(
        "train",
        training_result.metrics,
    )
    trainer.save_metrics(
        "train",
        training_result.metrics,
    )
    trainer.save_state()

    # ---------------------------------------------------------
    # Evaluate after training
    # ---------------------------------------------------------

    print("\nEvaluating trained adapter...")

    final_evaluation = trainer.evaluate(
        metric_key_prefix="after_training"
    )

    final_loss = final_evaluation.get(
        "after_training_loss"
    )

    final_perplexity = (
        math.exp(final_loss)
        if final_loss is not None and final_loss < 100
        else float("inf")
    )

    print(f"Final evaluation loss:         {final_loss}")
    print(f"Final evaluation perplexity:   {final_perplexity}")

    if initial_loss is not None and final_loss is not None:
        loss_change = final_loss - initial_loss
        perplexity_change = (
            final_perplexity - initial_perplexity
        )

        print(f"Evaluation loss change:        {loss_change:+.6f}")
        print(
            "Evaluation perplexity change:  "
            f"{perplexity_change:+.6f}"
        )

        if final_loss < initial_loss:
            print(
                "Verdict: evaluation loss improved after training."
            )
        elif final_loss > initial_loss:
            print(
                "Verdict: evaluation loss degraded after training."
            )
        else:
            print("Verdict: evaluation loss did not change.")

    trainer.log_metrics(
        "after_training",
        final_evaluation,
    )
    trainer.save_metrics(
        "after_training",
        final_evaluation,
    )

    # ---------------------------------------------------------
    # Save final adapter
    # ---------------------------------------------------------

    final_adapter_dir = output_dir / "final_adapter"

    trainer.save_model(str(final_adapter_dir))
    tokenizer.save_pretrained(str(final_adapter_dir))

    print("\nTraining complete.")
    print(f"Final adapter saved to: {final_adapter_dir}")

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "initial_evaluation": initial_evaluation,
        "initial_perplexity": initial_perplexity,
        "training_metrics": training_result.metrics,
        "final_evaluation": final_evaluation,
        "final_perplexity": final_perplexity,
        "adapter_directory": str(final_adapter_dir),
    }


### Saved Run Reporting

The following reporting helper reads each adapter's saved configuration, trainer state and evaluation files. It prints the trainable parameter count, LoRA settings, training hyperparameters, per-epoch validation loss, perplexity change, runtime and saved-artifact inventory without rerunning training.

In [8]:
import json
import math
import textwrap
from pathlib import Path

import torch


def print_qlora_metrics(run_path):
    """
    Print a complete, readable QLoRA run report from saved artifacts.

    Expected layout:

        run_path/
        ├── before_training_results.json
        ├── train_results.json
        ├── after_training_results.json
        ├── trainer_state.json
        ├── training_args.bin              # may also be in final_adapter/
        └── final_adapter/
            ├── adapter_config.json
            ├── adapter_model.safetensors
            └── ...

    Security:
        training_args.bin is a Python pickle-backed PyTorch file.
        Only use this function with artifacts you created or trust.
    """

    run_path = Path(run_path).expanduser().resolve()

    if not run_path.is_dir():
        raise FileNotFoundError(
            f"QLoRA run directory does not exist: {run_path}"
        )

    # ---------------------------------------------------------
    # File helpers
    # ---------------------------------------------------------

    def find_file(filename, prefer_final_adapter=False):
        preferred_paths = []

        if prefer_final_adapter:
            preferred_paths.extend([
                run_path / "final_adapter" / filename,
                run_path / filename,
            ])
        else:
            preferred_paths.extend([
                run_path / filename,
                run_path / "final_adapter" / filename,
            ])

        for candidate in preferred_paths:
            if candidate.is_file():
                return candidate

        matches = sorted(
            run_path.rglob(filename),
            key=lambda path: (
                "checkpoint-" in str(path),
                len(path.parts),
                str(path),
            ),
        )

        return matches[0] if matches else None

    def read_json(filename, prefer_final_adapter=False):
        path = find_file(
            filename,
            prefer_final_adapter=prefer_final_adapter,
        )

        if path is None:
            return None, None

        try:
            with path.open("r", encoding="utf-8") as file:
                return json.load(file), path
        except (OSError, json.JSONDecodeError) as error:
            print(f"Could not read {path}: {error}")
            return None, path

    def display_value(value):
        if value is None:
            return "Not recorded"

        if hasattr(value, "value"):
            value = value.value

        if isinstance(value, bool):
            return "Yes" if value else "No"

        if isinstance(value, (list, tuple, set)):
            return ", ".join(str(item) for item in value)

        if isinstance(value, dict):
            return json.dumps(value, sort_keys=True)

        if isinstance(value, float):
            if abs(value) >= 1_000_000:
                return f"{value:,.0f}"
            return f"{value:.6g}"

        return str(value)

    def print_section(title, rows, key_width=38, value_width=76):
        total_width = key_width + value_width + 3

        print("\n" + "=" * total_width)
        print(title)
        print("=" * total_width)

        for key, value in rows:
            key_lines = textwrap.wrap(
                str(key),
                width=key_width,
            ) or [""]

            value_lines = textwrap.wrap(
                display_value(value),
                width=value_width,
                break_long_words=False,
                break_on_hyphens=False,
            ) or [""]

            line_count = max(len(key_lines), len(value_lines))

            for index in range(line_count):
                key_part = (
                    key_lines[index]
                    if index < len(key_lines)
                    else ""
                )
                value_part = (
                    value_lines[index]
                    if index < len(value_lines)
                    else ""
                )

                print(
                    f"{key_part:<{key_width}} | {value_part}"
                )

    def safe_perplexity(loss):
        if loss is None:
            return None

        try:
            loss = float(loss)
            return math.exp(loss) if loss < 100 else float("inf")
        except (TypeError, ValueError, OverflowError):
            return None

    def get_argument(arguments, name, default=None):
        if arguments is None:
            return default
        return getattr(arguments, name, default)

    # ---------------------------------------------------------
    # Load JSON artifacts
    # ---------------------------------------------------------

    adapter_config, adapter_config_path = read_json(
        "adapter_config.json",
        prefer_final_adapter=True,
    )

    trainer_state, trainer_state_path = read_json(
        "trainer_state.json"
    )

    before_results, before_results_path = read_json(
        "before_training_results.json"
    )

    train_results, train_results_path = read_json(
        "train_results.json"
    )

    after_results, after_results_path = read_json(
        "after_training_results.json"
    )

    combined_report, combined_report_path = read_json(
        "qlora_metrics_report.json"
    )

    model_config, model_config_path = read_json(
        "config.json",
        prefer_final_adapter=True,
    )

    # ---------------------------------------------------------
    # Load training_args.bin
    # ---------------------------------------------------------

    training_args_path = find_file(
        "training_args.bin",
        prefer_final_adapter=True,
    )

    training_args = None
    training_args_error = None

    if training_args_path is not None:
        try:
            # weights_only=False is required because TrainingArguments
            # is a serialized Python object, not simply tensor weights.
            training_args = torch.load(
                training_args_path,
                map_location="cpu",
                weights_only=False,
            )
        except TypeError:
            # Compatibility with older PyTorch versions that do not
            # support the weights_only argument.
            try:
                training_args = torch.load(
                    training_args_path,
                    map_location="cpu",
                )
            except Exception as error:
                training_args_error = str(error)
        except Exception as error:
            training_args_error = str(error)

    # ---------------------------------------------------------
    # Resolve metric values
    # ---------------------------------------------------------

    before_results = before_results or {}
    train_results = train_results or {}
    after_results = after_results or {}
    trainer_state = trainer_state or {}
    adapter_config = adapter_config or {}

    initial_loss = before_results.get("before_training_loss")
    final_loss = after_results.get("after_training_loss")

    if combined_report:
        summary = combined_report.get("summary", {})

        if initial_loss is None:
            initial_loss = summary.get("initial_loss")

        if final_loss is None:
            final_loss = summary.get("final_loss")

    initial_perplexity = safe_perplexity(initial_loss)
    final_perplexity = safe_perplexity(final_loss)

    loss_change = None
    loss_change_percent = None
    perplexity_change = None
    perplexity_change_percent = None

    if initial_loss is not None and final_loss is not None:
        loss_change = final_loss - initial_loss

        if initial_loss != 0:
            loss_change_percent = (
                loss_change / initial_loss
            ) * 100

    if (
        initial_perplexity is not None
        and final_perplexity is not None
    ):
        perplexity_change = (
            final_perplexity - initial_perplexity
        )

        if initial_perplexity != 0:
            perplexity_change_percent = (
                perplexity_change / initial_perplexity
            ) * 100

    # ---------------------------------------------------------
    # Header
    # ---------------------------------------------------------

    print("\n" + "#" * 117)
    print("QLoRA RUN REPORT")
    print("#" * 117)
    print(f"Run directory: {run_path}")

    # ---------------------------------------------------------
    # Artifact inventory
    # ---------------------------------------------------------

    artifact_paths = [
        adapter_config_path,
        training_args_path,
        trainer_state_path,
        before_results_path,
        train_results_path,
        after_results_path,
        combined_report_path,
        model_config_path,
    ]

    adapter_weights_path = (
        find_file(
            "adapter_model.safetensors",
            prefer_final_adapter=True,
        )
        or find_file(
            "adapter_model.bin",
            prefer_final_adapter=True,
        )
    )

    if adapter_weights_path:
        artifact_paths.append(adapter_weights_path)

    artifact_rows = []

    for path in artifact_paths:
        if path is None:
            continue

        try:
            relative_path = path.relative_to(run_path)
        except ValueError:
            relative_path = path

        size_mb = path.stat().st_size / (1024 * 1024)

        artifact_rows.append(
            (
                str(relative_path),
                f"Present ({size_mb:.2f} MB)",
            )
        )

    print_section(
        "SAVED ARTIFACTS",
        artifact_rows or [("Artifacts", "None found")],
    )

    # ---------------------------------------------------------
    # Adapter configuration
    # ---------------------------------------------------------

    rank = adapter_config.get("r")
    alpha = adapter_config.get("lora_alpha")

    scaling = None
    if rank and alpha is not None:
        use_rslora = adapter_config.get(
            "use_rslora",
            False,
        )

        scaling = (
            alpha / math.sqrt(rank)
            if use_rslora
            else alpha / rank
        )

    print_section(
        "ADAPTER CONFIGURATION",
        [
            (
                "Base model/CPT checkpoint",
                adapter_config.get(
                    "base_model_name_or_path"
                ),
            ),
            (
                "PEFT type",
                adapter_config.get("peft_type"),
            ),
            (
                "Task type",
                adapter_config.get("task_type"),
            ),
            ("LoRA rank", rank),
            ("LoRA alpha", alpha),
            ("LoRA scaling", scaling),
            (
                "LoRA dropout",
                adapter_config.get("lora_dropout"),
            ),
            (
                "Target modules",
                adapter_config.get("target_modules"),
            ),
            (
                "Bias",
                adapter_config.get("bias"),
            ),
            (
                "Rank-stabilized LoRA",
                adapter_config.get("use_rslora"),
            ),
            (
                "Inference mode",
                adapter_config.get("inference_mode"),
            ),
            (
                "Modules additionally saved",
                adapter_config.get("modules_to_save"),
            ),
        ],
    )

    # ---------------------------------------------------------
    # Quantization configuration, when available
    # ---------------------------------------------------------

    quantization_config = None

    if model_config:
        quantization_config = model_config.get(
            "quantization_config"
        )

    if quantization_config:
        print_section(
            "QUANTIZATION CONFIGURATION",
            [
                (
                    key.replace("_", " ").title(),
                    value,
                )
                for key, value in quantization_config.items()
            ],
        )
    else:
        print_section(
            "QUANTIZATION CONFIGURATION",
            [
                (
                    "Configuration",
                    "Not present in the saved JSON/configuration files",
                ),
                (
                    "Expected for this QLoRA implementation",
                    "4-bit NF4; verify against the notebook run configuration",
                ),
            ],
        )

    # ---------------------------------------------------------
    # Training arguments
    # ---------------------------------------------------------

    if training_args is not None:
        train_batch_size = get_argument(
            training_args,
            "per_device_train_batch_size",
        )

        accumulation_steps = get_argument(
            training_args,
            "gradient_accumulation_steps",
        )

        effective_batch_size = None

        if (
            train_batch_size is not None
            and accumulation_steps is not None
        ):
            effective_batch_size = (
                train_batch_size * accumulation_steps
            )

        print_section(
            "TRAINING CONFIGURATION",
            [
                (
                    "Epochs",
                    get_argument(
                        training_args,
                        "num_train_epochs",
                    ),
                ),
                (
                    "Maximum optimizer steps",
                    get_argument(training_args, "max_steps"),
                ),
                (
                    "Learning rate",
                    get_argument(
                        training_args,
                        "learning_rate",
                    ),
                ),
                (
                    "Per-device training batch size",
                    train_batch_size,
                ),
                (
                    "Per-device evaluation batch size",
                    get_argument(
                        training_args,
                        "per_device_eval_batch_size",
                    ),
                ),
                (
                    "Gradient accumulation steps",
                    accumulation_steps,
                ),
                (
                    "Effective batch size on one GPU",
                    effective_batch_size,
                ),
                (
                    "Maximum sequence length",
                    get_argument(
                        training_args,
                        "max_seq_length",
                    ),
                ),
                (
                    "Packing",
                    get_argument(training_args, "packing"),
                ),
                (
                    "Optimizer",
                    get_argument(training_args, "optim"),
                ),
                (
                    "Learning-rate scheduler",
                    get_argument(
                        training_args,
                        "lr_scheduler_type",
                    ),
                ),
                (
                    "Warmup ratio",
                    get_argument(
                        training_args,
                        "warmup_ratio",
                    ),
                ),
                (
                    "Warmup steps",
                    get_argument(
                        training_args,
                        "warmup_steps",
                    ),
                ),
                (
                    "Weight decay",
                    get_argument(
                        training_args,
                        "weight_decay",
                    ),
                ),
                (
                    "Maximum gradient norm",
                    get_argument(
                        training_args,
                        "max_grad_norm",
                    ),
                ),
                (
                    "BF16",
                    get_argument(training_args, "bf16"),
                ),
                (
                    "FP16",
                    get_argument(training_args, "fp16"),
                ),
                (
                    "TF32",
                    get_argument(training_args, "tf32"),
                ),
                (
                    "Gradient checkpointing",
                    get_argument(
                        training_args,
                        "gradient_checkpointing",
                    ),
                ),
                (
                    "Evaluation strategy",
                    get_argument(
                        training_args,
                        "eval_strategy",
                        get_argument(
                            training_args,
                            "evaluation_strategy",
                        ),
                    ),
                ),
                (
                    "Save strategy",
                    get_argument(
                        training_args,
                        "save_strategy",
                    ),
                ),
                (
                    "Logging strategy",
                    get_argument(
                        training_args,
                        "logging_strategy",
                    ),
                ),
                (
                    "Logging steps",
                    get_argument(
                        training_args,
                        "logging_steps",
                    ),
                ),
                (
                    "Save total limit",
                    get_argument(
                        training_args,
                        "save_total_limit",
                    ),
                ),
                (
                    "Load best model at end",
                    get_argument(
                        training_args,
                        "load_best_model_at_end",
                    ),
                ),
                (
                    "Best-model metric",
                    get_argument(
                        training_args,
                        "metric_for_best_model",
                    ),
                ),
                (
                    "Higher metric is better",
                    get_argument(
                        training_args,
                        "greater_is_better",
                    ),
                ),
                (
                    "Random seed",
                    get_argument(training_args, "seed"),
                ),
                (
                    "Data seed",
                    get_argument(
                        training_args,
                        "data_seed",
                    ),
                ),
                (
                    "Data-loader workers",
                    get_argument(
                        training_args,
                        "dataloader_num_workers",
                    ),
                ),
                (
                    "Progress bar disabled",
                    get_argument(
                        training_args,
                        "disable_tqdm",
                    ),
                ),
                (
                    "Configured output directory",
                    get_argument(
                        training_args,
                        "output_dir",
                    ),
                ),
            ],
        )
    else:
        reason = (
            training_args_error
            if training_args_error
            else "training_args.bin was not found"
        )

        print_section(
            "TRAINING CONFIGURATION",
            [
                ("Status", "Unavailable"),
                ("Reason", reason),
            ],
        )

    # ---------------------------------------------------------
    # Trainer state
    # ---------------------------------------------------------

    print_section(
        "TRAINER STATE",
        [
            (
                "Completed epochs",
                trainer_state.get("epoch"),
            ),
            (
                "Global optimizer steps",
                trainer_state.get("global_step"),
            ),
            (
                "Maximum optimizer steps",
                trainer_state.get("max_steps"),
            ),
            (
                "Logging interval",
                trainer_state.get("logging_steps"),
            ),
            (
                "Saved checkpoint interval",
                trainer_state.get("save_steps"),
            ),
            (
                "Best metric",
                trainer_state.get("best_metric"),
            ),
            (
                "Best model checkpoint",
                trainer_state.get(
                    "best_model_checkpoint"
                ),
            ),
            (
                "Total floating-point operations",
                trainer_state.get("total_flos"),
            ),
        ],
    )

    # ---------------------------------------------------------
    # Evaluation summary
    # ---------------------------------------------------------

    if (
        initial_loss is not None
        and final_loss is not None
        and final_loss < initial_loss
    ):
        verdict = "GOOD — held-out evaluation performance improved"
    elif (
        initial_loss is not None
        and final_loss is not None
        and final_loss > initial_loss
    ):
        verdict = (
            "NEEDS IMPROVEMENT — held-out evaluation "
            "performance degraded"
        )
    elif initial_loss is not None and final_loss is not None:
        verdict = "UNCHANGED — no measurable difference"
    else:
        verdict = "UNAVAILABLE — evaluation metrics are incomplete"

    print_section(
        "EVALUATION SUMMARY",
        [
            ("Initial evaluation loss", initial_loss),
            ("Final evaluation loss", final_loss),
            ("Loss change", loss_change),
            ("Loss change percent", loss_change_percent),
            (
                "Initial evaluation perplexity",
                initial_perplexity,
            ),
            (
                "Final evaluation perplexity",
                final_perplexity,
            ),
            (
                "Perplexity change",
                perplexity_change,
            ),
            (
                "Perplexity change percent",
                perplexity_change_percent,
            ),
            ("Verdict", verdict),
        ],
    )

    # ---------------------------------------------------------
    # Per-epoch validation history
    # ---------------------------------------------------------

    log_history = trainer_state.get("log_history", [])

    if not log_history and combined_report:
        log_history = combined_report.get(
            "training_history",
            [],
        )

    epoch_evaluations = [
        entry
        for entry in log_history
        if "eval_loss" in entry
    ]

    if epoch_evaluations:
        print("\n" + "=" * 86)
        print("PER-EPOCH VALIDATION")
        print("=" * 86)
        print(
            f"{'Epoch':>8} | "
            f"{'Step':>8} | "
            f"{'Validation loss':>18} | "
            f"{'Perplexity':>14} | "
            f"{'Status':<15}"
        )
        print("-" * 86)

        best_entry = min(
            epoch_evaluations,
            key=lambda entry: entry["eval_loss"],
        )

        for entry in epoch_evaluations:
            eval_loss = entry["eval_loss"]
            eval_perplexity = safe_perplexity(eval_loss)

            status = (
                "BEST"
                if entry is best_entry
                else ""
            )

            print(
                f"{entry.get('epoch', 0):>8.2f} | "
                f"{entry.get('step', 0):>8} | "
                f"{eval_loss:>18.6f} | "
                f"{eval_perplexity:>14.4f} | "
                f"{status:<15}"
            )

    # ---------------------------------------------------------
    # Training performance
    # ---------------------------------------------------------

    print_section(
        "TRAINING PERFORMANCE",
        [
            (
                "Final training loss",
                train_results.get("train_loss"),
            ),
            (
                "Training runtime in seconds",
                train_results.get("train_runtime"),
            ),
            (
                "Training samples per second",
                train_results.get(
                    "train_samples_per_second"
                ),
            ),
            (
                "Training steps per second",
                train_results.get(
                    "train_steps_per_second"
                ),
            ),
            (
                "Completed epochs",
                train_results.get("epoch"),
            ),
            (
                "Total floating-point operations",
                (
                    combined_report
                    and combined_report
                    .get("training", {})
                    .get("total_flos")
                ),
            ),
        ],
    )

    # ---------------------------------------------------------
    # Information not normally persisted
    # ---------------------------------------------------------

    print_section(
        "REPRODUCIBILITY NOTES",
        [
            (
                "Dataset paths and example counts",
                "Not stored by Trainer; retain them in the notebook",
            ),
            (
                "4-bit quantization settings",
                (
                    "Available only if config.json contains "
                    "quantization_config; otherwise retain the notebook print"
                ),
            ),
            (
                "GPU hardware",
                "Not stored by Trainer; retain the notebook output",
            ),
            (
                "Software versions",
                (
                    "Not stored here; record torch, transformers, "
                    "trl, peft, bitsandbytes and CUDA versions"
                ),
            ),
        ],
    )

    print("\n" + "#" * 117)
    print("END OF QLoRA RUN REPORT")
    print("#" * 117)



<a id="adapter-a"></a>
### Adapter A Low Capacity

The following cell trains the low-capacity adapter with rank 8, alpha 16 and LoRA updates on `q_proj` and `v_proj`. This is the smallest and least expensive configuration; it provides the baseline against which the additional capacity of Adapters B and C is measured. The saved run report follows the training output.

In [9]:


# Adapter A
result_a = run_qlora_finetuning_with_chat_template(
    model_name=CPT_MODEL_DIR,
    train_file=INSTRUCTION_DATA_DIR / "train.jsonl",
    evaluation_file=INSTRUCTION_DATA_DIR / "evaluation.jsonl",
    output_dir=RUN_OUTPUT_DIR / "adapter-a",
    lora_rank=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
)

print_qlora_metrics(RUN_OUTPUT_DIR / "adapter-a")


QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model
Training data:                /home/jovyan/llmassign/1a/data/lora-instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/lora-instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/adapter-a
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    8
LoRA alpha:                   16
LoRA scaling:                 2.00
LoRA dropout:                 0.05
Target modules:               ['q_proj', 'v_proj']
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
Progress bar disabled:        True
EOS token: '<|endoftext|>' (ID: 151643)
PAD token:

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Formatted training example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What operational role does Cisco Nexus Dashboard provide for on-premises data centers?<|im_end|>
<|im_start|>assistant
Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations.<|im_end|>



Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 31
Text contributing to loss: Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations.<|im_end|>

Validated batch shape: (2, 102)

Evaluating before adapter training...
{'before_training_loss': 3.538893222808838, 'before_training_model_preparation_time': 0.005, 'before_training_runtime': 3.4491, 'before_training_samples_per_second': 11.597, 'before_training_steps_per_second': 11.597}
Initial evaluation loss:       3.538893222808838
Initial evaluation perplexity: 34.42879309328681

Starting QLoRA training...
{'loss': 3.8368, 'grad_norm': 2.2557930946350098, 'learning_rate': 5e-05, 'epoch': 0.05}
{'loss': 3.4664, 'grad_norm': 1.2527389526367188, 'learning_rate': 9.482758620689656e-05, 'epoch': 0.25}
{'loss': 3.4771, 'grad_norm': 1.579913854598999, 'learning_rate': 8.620689655172413e-05, 'epoch': 0.5}
{'loss': 3.2958, 'grad_norm': 1.781949

<a id="adapter-b"></a>
### Adapter B Balanced

The following cell trains the balanced adapter with rank 16, alpha 32 and the same `q_proj` and `v_proj` targets. Doubling the rank doubles the trainable adapter capacity while preserving the same LoRA scaling factor as Adapter A. Its metrics are printed in the next cell.

In [10]:
# Adapter B — balanced
result_b = run_qlora_finetuning_with_chat_template(
    model_name=CPT_MODEL_DIR,
    train_file=INSTRUCTION_DATA_DIR / "train.jsonl",
    evaluation_file=INSTRUCTION_DATA_DIR / "evaluation.jsonl",
    output_dir=RUN_OUTPUT_DIR / "adapter-b",
    lora_rank=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
)



QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model
Training data:                /home/jovyan/llmassign/1a/data/lora-instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/lora-instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/adapter-b
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    16
LoRA alpha:                   32
LoRA scaling:                 2.00
LoRA dropout:                 0.05
Target modules:               ['q_proj', 'v_proj']
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
Progress bar disabled:        True
EOS token: '<|endoftext|>' (ID: 151643)
PAD token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
Training examples:            160
Evaluation examples:          40


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Formatted training example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What operational role does Cisco Nexus Dashboard provide for on-premises data centers?<|im_end|>
<|im_start|>assistant
Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations.<|im_end|>



Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 31
Text contributing to loss: Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations.<|im_end|>

Validated batch shape: (2, 102)

Evaluating before adapter training...
{'before_training_loss': 3.538893222808838, 'before_training_model_preparation_time': 0.0028, 'before_training_runtime': 3.0377, 'before_training_samples_per_second': 13.168, 'before_training_steps_per_second': 13.168}
Initial evaluation loss:       3.538893222808838
Initial evaluation perplexity: 34.42879309328681

Starting QLoRA training...
{'loss': 3.8368, 'grad_norm': 3.1268601417541504, 'learning_rate': 5e-05, 'epoch': 0.05}
{'loss': 3.457, 'grad_norm': 1.2790215015411377, 'learning_rate': 9.482758620689656e-05, 'epoch': 0.25}
{'loss': 3.4278, 'grad_norm': 1.562452793121338, 'learning_rate': 8.620689655172413e-05, 'epoch': 0.5}
{'loss': 3.1963, 'grad_norm': 1.694755

In [11]:
print_qlora_metrics(RUN_OUTPUT_DIR / "adapter-b")


#####################################################################################################################
QLoRA RUN REPORT
#####################################################################################################################
Run directory: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/adapter-b

SAVED ARTIFACTS
final_adapter/adapter_config.json      | Present (0.00 MB)
final_adapter/training_args.bin        | Present (0.01 MB)
trainer_state.json                     | Present (0.00 MB)
before_training_results.json           | Present (0.00 MB)
train_results.json                     | Present (0.00 MB)
after_training_results.json            | Present (0.00 MB)
final_adapter/adapter_model.safetensor | Present (8.33 MB)
s                                      | 

ADAPTER CONFIGURATION
Base model/CPT checkpoint              | /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model
PEFT type                              | LORA
Task type                    

<a id="adapter-c"></a>
### Adapter C High Capacity

The following cell trains the high-capacity adapter with rank 32 and alpha 32. In addition to `q_proj` and `v_proj`, it adapts `o_proj`, increasing the number of trainable parameters and runtime. Its saved metrics show whether this extra capacity improves held-out prediction and generated answers.

In [12]:
# Adapter C — high capacity
result_c = run_qlora_finetuning_with_chat_template(
    model_name=CPT_MODEL_DIR,
    train_file=INSTRUCTION_DATA_DIR / "train.jsonl",
    evaluation_file=INSTRUCTION_DATA_DIR / "evaluation.jsonl",
    output_dir=RUN_OUTPUT_DIR / "adapter-c",
    lora_rank=32,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj"],
)



QLoRA TRAINING CONFIGURATION
Model/CPT checkpoint:         /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model
Training data:                /home/jovyan/llmassign/1a/data/lora-instructions/train.jsonl
Evaluation data:              /home/jovyan/llmassign/1a/data/lora-instructions/evaluation.jsonl
Output directory:             /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/adapter-c
GPU:                          NVIDIA A100-SXM4-80GB
Compute dtype:                torch.bfloat16
TF32 enabled:                 True
LoRA rank:                    32
LoRA alpha:                   32
LoRA scaling:                 1.00
LoRA dropout:                 0.05
Target modules:               ['q_proj', 'v_proj', 'o_proj']
Epochs:                       3
Learning rate:                0.0001
Micro-batch size:             1
Gradient accumulation:        8
Effective batch size:          8
Maximum sequence length:      1024
Progress bar disabled:        True
EOS token: '<|endoftext|>' (ID: 151643)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 7,110,656 || all params: 1,550,824,960 || trainable%: 0.4585
Training examples:            160
Evaluation examples:          40


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Formatted training example:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What operational role does Cisco Nexus Dashboard provide for on-premises data centers?<|im_end|>
<|im_start|>assistant
Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations.<|im_end|>



Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokens contributing to loss: 31
Text contributing to loss: Nexus Dashboard provides a centralized operating platform for provisioning, visibility, troubleshooting, automation, and analytics across switches, fabrics, sites, and locations.<|im_end|>

Validated batch shape: (2, 102)

Evaluating before adapter training...
{'before_training_loss': 3.538893222808838, 'before_training_model_preparation_time': 0.0042, 'before_training_runtime': 3.1792, 'before_training_samples_per_second': 12.582, 'before_training_steps_per_second': 12.582}
Initial evaluation loss:       3.538893222808838
Initial evaluation perplexity: 34.42879309328681

Starting QLoRA training...
{'loss': 3.8368, 'grad_norm': 2.7051546573638916, 'learning_rate': 5e-05, 'epoch': 0.05}
{'loss': 3.4462, 'grad_norm': 1.018206000328064, 'learning_rate': 9.482758620689656e-05, 'epoch': 0.25}
{'loss': 3.3554, 'grad_norm': 1.3394132852554321, 'learning_rate': 8.620689655172413e-05, 'epoch': 0.5}
{'loss': 3.0927, 'grad_norm': 1.27281

In [13]:
print_qlora_metrics(RUN_OUTPUT_DIR / "adapter-c")


#####################################################################################################################
QLoRA RUN REPORT
#####################################################################################################################
Run directory: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/adapter-c

SAVED ARTIFACTS
final_adapter/adapter_config.json      | Present (0.00 MB)
final_adapter/training_args.bin        | Present (0.01 MB)
trainer_state.json                     | Present (0.00 MB)
before_training_results.json           | Present (0.00 MB)
train_results.json                     | Present (0.00 MB)
after_training_results.json            | Present (0.00 MB)
final_adapter/adapter_model.safetensor | Present (27.15 MB)
s                                      | 

ADAPTER CONFIGURATION
Base model/CPT checkpoint              | /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model
PEFT type                              | LORA
Task type                   

<a id="part-b3-comparative-evaluation"></a>
## Part B3 Comparative Evaluation

Evaluation uses two complementary views. First, the saved run reports compare adapter configuration, held-out response-only loss, perplexity and training cost. Second, every adapter generates an answer for the same 40 held-out instructions using greedy decoding. Each generated answer is compared with its grounded reference response using token-level F1, and the notebook displays the same selected prompts side by side across Adapters A, B and C.

Token F1 is a reproducible measure of reference overlap rather than a complete factuality judge. The final recommendation therefore considers both the aggregate F1 results and qualitative domain relevance of the displayed responses, while validation perplexity is reported separately as a teacher-forced prediction metric.

In [14]:
#!/usr/bin/env python3
"""Print an assignment-ready comparison of saved QLoRA adapter runs."""

from __future__ import annotations

import argparse
import json
import math
import sys
import textwrap
from pathlib import Path
from typing import Any


ADAPTER_PATTERNS = ("*adapter*", "*adaptor*")


def _read_json(path: Path | None) -> dict[str, Any]:
    if path is None or not path.is_file():
        return {}
    try:
        with path.open("r", encoding="utf-8") as handle:
            value = json.load(handle)
    except (OSError, json.JSONDecodeError) as error:
        print(f"warning: could not read {path}: {error}", file=sys.stderr)
        return {}
    return value if isinstance(value, dict) else {}


def _find_file(run_dir: Path, filename: str) -> Path | None:
    """Prefer final artifacts over checkpoint copies of the same file."""
    for candidate in (
        run_dir / filename,
        run_dir / "final_adapter" / filename,
    ):
        if candidate.is_file():
            return candidate

    matches = sorted(
        run_dir.rglob(filename),
        key=lambda path: (
            "checkpoint-" in str(path),
            len(path.parts),
            str(path),
        ),
    )
    return matches[0] if matches else None


def _metric(
    primary: dict[str, Any],
    key: str,
    report: dict[str, Any],
    report_key: str,
) -> float | None:
    value = primary.get(key)
    if value is None:
        value = report.get("summary", {}).get(report_key)
    try:
        return float(value) if value is not None else None
    except (TypeError, ValueError):
        return None


def _perplexity(loss: float | None) -> float | None:
    if loss is None:
        return None
    try:
        return math.exp(loss) if loss < 100 else float("inf")
    except OverflowError:
        return float("inf")


def _number(value: Any) -> float | None:
    try:
        return float(value) if value is not None else None
    except (TypeError, ValueError):
        return None


def _target_modules(value: Any) -> str:
    if isinstance(value, (list, tuple, set)):
        return ", ".join(sorted(str(item) for item in value))
    return str(value) if value is not None else "Not recorded"


def _best_epoch(state: dict[str, Any], report: dict[str, Any]) -> tuple[Any, Any]:
    history = state.get("log_history") or report.get("training_history") or []
    evaluations = [
        item
        for item in history
        if isinstance(item, dict) and _number(item.get("eval_loss")) is not None
    ]
    if not evaluations:
        return None, None
    best = min(evaluations, key=lambda item: float(item["eval_loss"]))
    return best.get("epoch"), float(best["eval_loss"])


def _capacity_label(rank: Any) -> str:
    rank_number = _number(rank)
    if rank_number == 8:
        return "A / Low"
    if rank_number == 16:
        return "B / Balanced"
    if rank_number == 32:
        return "C / High"
    return "Custom"


def _load_run(run_dir: Path) -> dict[str, Any]:
    adapter = _read_json(_find_file(run_dir, "adapter_config.json"))
    before = _read_json(_find_file(run_dir, "before_training_results.json"))
    after = _read_json(_find_file(run_dir, "after_training_results.json"))
    training = _read_json(_find_file(run_dir, "train_results.json"))
    state = _read_json(_find_file(run_dir, "trainer_state.json"))
    report = _read_json(_find_file(run_dir, "qlora_metrics_report.json"))

    initial_loss = _metric(
        before,
        "before_training_loss",
        report,
        "initial_loss",
    )
    final_loss = _metric(
        after,
        "after_training_loss",
        report,
        "final_loss",
    )
    initial_ppl = _perplexity(initial_loss)
    final_ppl = _perplexity(final_loss)

    ppl_change_percent = None
    if initial_ppl not in (None, 0) and final_ppl is not None:
        ppl_change_percent = ((final_ppl - initial_ppl) / initial_ppl) * 100

    best_epoch, best_eval_loss = _best_epoch(state, report)
    rank = adapter.get("r")

    return {
        "directory": run_dir.name,
        "capacity": _capacity_label(rank),
        "base_model": adapter.get("base_model_name_or_path"),
        "rank": rank,
        "alpha": adapter.get("lora_alpha"),
        "dropout": adapter.get("lora_dropout"),
        "targets": _target_modules(adapter.get("target_modules")),
        "initial_loss": initial_loss,
        "final_loss": final_loss,
        "initial_ppl": initial_ppl,
        "final_ppl": final_ppl,
        "ppl_change_percent": ppl_change_percent,
        "train_loss": _number(training.get("train_loss")),
        "runtime": _number(training.get("train_runtime")),
        "epochs": _number(training.get("epoch") or state.get("epoch")),
        "best_epoch": best_epoch,
        "best_eval_loss": best_eval_loss,
    }


def _discover_runs(parent: Path) -> list[Path]:
    found: dict[Path, None] = {}
    for pattern in ADAPTER_PATTERNS:
        for path in parent.glob(pattern):
            if path.is_dir():
                found[path.resolve()] = None
    return sorted(found, key=lambda path: path.name.lower())


def _format(value: Any, kind: str = "text") -> str:
    if value is None:
        return "-"
    if kind == "loss":
        return f"{float(value):.4f}"
    if kind == "ppl":
        return f"{float(value):.3f}"
    if kind == "percent":
        return f"{float(value):+.2f}%"
    if kind == "seconds":
        return f"{float(value):.1f}s"
    if kind == "epoch":
        return f"{float(value):.2f}"
    return str(value)


def _print_wrapped_table(
    headers: list[str],
    rows: list[list[str]],
    widths: list[int],
) -> None:
    def separator(character: str = "-") -> str:
        return "+" + "+".join(character * (width + 2) for width in widths) + "+"

    def wrapped_row(row: list[str]) -> None:
        cells = [
            textwrap.wrap(
                str(value),
                width=width,
                break_long_words=False,
                break_on_hyphens=False,
            )
            or [""]
            for value, width in zip(row, widths)
        ]
        for line_number in range(max(len(cell) for cell in cells)):
            values = [
                cell[line_number] if line_number < len(cell) else ""
                for cell in cells
            ]
            print(
                "| "
                + " | ".join(
                    f"{value:<{width}}" for value, width in zip(values, widths)
                )
                + " |"
            )

    print(separator("="))
    wrapped_row(headers)
    print(separator("="))
    for row in rows:
        wrapped_row(row)
        print(separator())


def print_adapter_comparison(parent_folder: str | Path) -> list[dict[str, Any]]:
    """Read every adapter/adaptor child directory and print comparison tables."""
    parent = Path(parent_folder).expanduser().resolve()
    if not parent.is_dir():
        raise FileNotFoundError(f"Parent folder does not exist: {parent}")

    run_dirs = _discover_runs(parent)
    if not run_dirs:
        patterns = ", ".join(ADAPTER_PATTERNS)
        raise FileNotFoundError(
            f"No adapter run directories matching {patterns} under {parent}"
        )

    runs = [_load_run(path) for path in run_dirs]
    runs.sort(
        key=lambda run: (
            _number(run["rank"]) is None,
            _number(run["rank"]) or float("inf"),
            run["directory"].lower(),
        )
    )
    comparable = [run for run in runs if run["final_loss"] is not None]
    best = min(comparable, key=lambda run: run["final_loss"]) if comparable else None

    print("\nQLoRA ADAPTER CONFIGURATION COMPARISON")
    print(f"Parent folder: {parent}\n")
    config_rows = []
    for run in runs:
        config_rows.append(
            [
                run["directory"],
                run["capacity"],
                _format(run["rank"]),
                _format(run["alpha"]),
                _format(run["dropout"]),
                run["targets"],
            ]
        )
    _print_wrapped_table(
        ["Run", "Assignment", "Rank", "Alpha", "Dropout", "Target modules"],
        config_rows,
        [20, 13, 6, 7, 9, 30],
    )

    print("\nHELD-OUT EVALUATION COMPARISON\n")
    metric_rows = []
    for run in runs:
        if best is run:
            verdict = "BEST"
        elif run["ppl_change_percent"] is None:
            verdict = "Incomplete"
        elif run["ppl_change_percent"] < 0:
            verdict = "Improved"
        else:
            verdict = "Degraded"

        metric_rows.append(
            [
                run["directory"],
                _format(run["initial_loss"], "loss"),
                _format(run["final_loss"], "loss"),
                _format(run["initial_ppl"], "ppl"),
                _format(run["final_ppl"], "ppl"),
                _format(run["ppl_change_percent"], "percent"),
                _format(run["best_epoch"], "epoch"),
                verdict,
            ]
        )
    _print_wrapped_table(
        [
            "Run",
            "Initial loss",
            "Final loss",
            "Initial PPL",
            "Final PPL",
            "PPL change",
            "Best epoch",
            "Verdict",
        ],
        metric_rows,
        [20, 12, 11, 11, 10, 11, 10, 10],
    )

    print("\nTRAINING COST COMPARISON\n")
    cost_rows = [
        [
            run["directory"],
            _format(run["epochs"], "epoch"),
            _format(run["train_loss"], "loss"),
            _format(run["runtime"], "seconds"),
            _format(run["best_eval_loss"], "loss"),
        ]
        for run in runs
    ]
    _print_wrapped_table(
        ["Run", "Epochs", "Train loss", "Runtime", "Best validation loss"],
        cost_rows,
        [20, 8, 12, 12, 22],
    )

    if best:
        print(
            "\nRecommended adapter by lowest held-out evaluation loss: "
            f"{best['directory']} (loss={best['final_loss']:.4f}, "
            f"PPL={best['final_ppl']:.3f})."
        )
    else:
        print("\nNo complete final evaluation metrics were found.")

    base_models = {run["base_model"] for run in runs if run["base_model"]}
    if len(base_models) > 1:
        print(
            "warning: adapters use different base-model paths, so their loss and "
            "perplexity values may not be directly comparable.",
            file=sys.stderr,
        )

    print(
        "\nNote: this compares adapter configuration, held-out loss, perplexity, "
        "and runtime. Assignment Part B3 also requires the baseline and all three "
        "adapters' generated answers for the same three domain prompts."
    )
    return runs


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Compare QLoRA runs stored in child folders whose names contain "
            "adapter or adaptor."
        )
    )
    parser.add_argument(
        "parent_folder",
        help="Folder containing the adapter run directories.",
    )
    return parser

import json
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def compare_all_adapters(
    base_model_folder,
    adapters_parent,
    evaluation_jsonl,
    output_json,
    max_new_tokens=200,
    batch_size=4,
    sample_size=10,
    sample_mode="best",       # "best" or "random"
    correctness_threshold=0.30,
    seed=42,
):
    base_model_folder = Path(base_model_folder)
    adapters_parent = Path(adapters_parent)
    output_json = Path(output_json)

    # Load held-out instruction/response pairs.
    with open(evaluation_jsonl, encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]

    # Find final adapters in folders containing adapter/adaptor.
    adapter_paths = {}
    for run_dir in sorted(adapters_parent.iterdir()):
        if not run_dir.is_dir():
            continue
        if "adapter" not in run_dir.name.lower() and "adaptor" not in run_dir.name.lower():
            continue

        candidate = run_dir / "final_adapter"
        if not (candidate / "adapter_config.json").is_file():
            candidate = run_dir

        if (candidate / "adapter_config.json").is_file():
            adapter_paths[run_dir.name] = candidate

    if not adapter_paths:
        raise ValueError(f"No adapters found under {adapters_parent}")

    dtype = (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_folder)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    # Only used when a tokenizer does not provide its own chat template.
    if not tokenizer.chat_template:
        tokenizer.chat_template = """
{% for message in messages %}
{% if message['role'] == 'user' %}User: {{ message['content'] }}
{% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }}
{% endif %}
{% endfor %}
{% if add_generation_prompt %}Assistant: {% endif %}
"""

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_folder,
        torch_dtype=dtype,
        device_map="auto",
    )

    # Load every adapter into the same base model.
    adapter_names = {}
    for index, (display_name, adapter_path) in enumerate(adapter_paths.items()):
        internal_name = f"adapter_{index}"
        adapter_names[display_name] = internal_name

        if index == 0:
            model = PeftModel.from_pretrained(
                base_model,
                adapter_path,
                adapter_name=internal_name,
                is_trainable=False,
            )
        else:
            model.load_adapter(
                adapter_path,
                adapter_name=internal_name,
                is_trainable=False,
            )

    model.eval()
    device = next(model.parameters()).device

    # Detect ordinary EOS plus a chat-template end-of-message token.
    stop_ids = []
    if tokenizer.eos_token_id is not None:
        stop_ids.append(tokenizer.eos_token_id)

    probe = [{"role": "user", "content": "x"}]
    prompt_ids = tokenizer.apply_chat_template(
        probe,
        tokenize=True,
        add_generation_prompt=True,
    )
    completed_ids = tokenizer.apply_chat_template(
        probe + [{"role": "assistant", "content": ""}],
        tokenize=True,
        add_generation_prompt=False,
    )

    if completed_ids[:len(prompt_ids)] == prompt_ids:
        for token_id in completed_ids[len(prompt_ids):]:
            if token_id in tokenizer.all_special_ids and token_id not in stop_ids:
                stop_ids.append(token_id)

    stop_ids = stop_ids or None

    def generate(adapter_name):
        model.set_adapter(adapter_name)
        generated_responses = []

        for start in range(0, len(records), batch_size):
            batch = records[start:start + batch_size]

            prompts = [
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": item["instruction"]}],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for item in batch
            ]

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                add_special_tokens=False,
            ).to(device)

            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    eos_token_id=stop_ids,
                    pad_token_id=tokenizer.pad_token_id,
                )

            prompt_width = inputs["input_ids"].shape[1]
            responses = tokenizer.batch_decode(
                generated[:, prompt_width:],
                skip_special_tokens=True,
            )
            generated_responses.extend(response.strip() for response in responses)

        return generated_responses

    def token_f1(prediction, reference):
        def tokens(text):
            return re.findall(r"[a-z0-9][a-z0-9_-]*", text.lower())

        predicted = Counter(tokens(prediction))
        expected = Counter(tokens(reference))

        if not predicted or not expected:
            return 0.0

        overlap = sum((predicted & expected).values())
        precision = overlap / sum(predicted.values())
        recall = overlap / sum(expected.values())

        return (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

    results = [
        {
            "instruction": item["instruction"],
            "reference_response": item["response"],
            "adapters": {},
        }
        for item in records
    ]

    # Generate and score every evaluation record with every adapter.
    for display_name, internal_name in adapter_names.items():
        responses = generate(internal_name)

        for result, response in zip(results, responses):
            score = token_f1(response, result["reference_response"])
            result["adapters"][display_name] = {
                "response": response,
                "correctness": {
                    "method": "reference_token_f1",
                    "score": round(score, 4),
                    "verdict": (
                        "likely_correct"
                        if score >= correctness_threshold
                        else "needs_review"
                    ),
                },
            }

    summary = {}
    for adapter_name in adapter_paths:
        scores = [
            result["adapters"][adapter_name]["correctness"]["score"]
            for result in results
        ]
        summary[adapter_name] = {
            "average_correctness": round(sum(scores) / len(scores), 4),
            "likely_correct": sum(
                score >= correctness_threshold for score in scores
            ),
            "evaluated": len(scores),
        }

    output = {
        "base_model": str(base_model_folder.resolve()),
        "evaluation_file": str(Path(evaluation_jsonl).resolve()),
        "correctness_method": "reference_token_f1",
        "correctness_threshold": correctness_threshold,
        "adapters": {
            name: str(path.resolve())
            for name, path in adapter_paths.items()
        },
        "summary": summary,
        "results": results,
    }

    output_json.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, "w", encoding="utf-8") as handle:
        json.dump(output, handle, indent=2, ensure_ascii=False)

    # Rank adapters using the complete evaluation set.
    overall_table = pd.DataFrame([
        {
            "Adapter": name,
            "Average F1": values["average_correctness"],
            "Likely correct": f"{values['likely_correct']}/{values['evaluated']}",
        }
        for name, values in summary.items()
    ]).sort_values("Average F1", ascending=False)

    print("\nOVERALL HELD-OUT EVALUATION")
    display(overall_table)

    # Select ten representative results for a readable comparison.
    if sample_mode == "random":
        selected = random.Random(seed).sample(
            results,
            min(sample_size, len(results)),
        )
    else:
        selected = sorted(
            results,
            key=lambda result: max(
                value["correctness"]["score"]
                for value in result["adapters"].values()
            ),
            reverse=True,
        )[:sample_size]

    comparison_rows = []
    for index, result in enumerate(selected, start=1):
        row = {
            "#": index,
            "Instruction": result["instruction"],
        }

        best_adapter = max(
            result["adapters"],
            key=lambda name: result["adapters"][name]["correctness"]["score"],
        )

        for adapter_name, value in result["adapters"].items():
            score = value["correctness"]["score"]
            response = value["response"].replace("\n", " ")
            row[adapter_name] = f"{score:.2f} | {response[:140]}"

        row["Best"] = best_adapter
        comparison_rows.append(row)

    print(f"\n{sample_mode.upper()} {len(selected)} RESPONSE COMPARISON")
    with pd.option_context("display.max_colwidth", 150):
        display(pd.DataFrame(comparison_rows))

    print(f"\nFull responses saved to: {output_json}")
    return output

import json
import random
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def print_adapter_inference_comparison(
    result_json,
    limit=10,
    selection="best",       # "best" or "random"
    seed=42,
    reference_preview_chars=500,
):
    with Path(result_json).open(encoding="utf-8") as handle:
        report = json.load(handle)

    results = report.get("results", [])
    if not results:
        raise ValueError("The result JSON contains no inference results.")

    adapter_names = sorted({
        adapter_name
        for result in results
        for adapter_name in result.get("adapters", {})
    })

    if not adapter_names:
        raise ValueError("No adapter responses were found in the result JSON.")

    def correctness_score(adapter_result):
        correctness = adapter_result.get("correctness", 0)

        if isinstance(correctness, dict):
            correctness = correctness.get("score", 0)

        try:
            return float(correctness)
        except (TypeError, ValueError):
            return 0.0

    def correctness_verdict(adapter_result):
        correctness = adapter_result.get("correctness", {})

        if isinstance(correctness, dict):
            return correctness.get("verdict", "not_recorded")

        return "not_recorded"

    # Rank adapters across the complete evaluation set.
    summary_rows = []

    for adapter_name in adapter_names:
        adapter_results = [
            result["adapters"][adapter_name]
            for result in results
            if adapter_name in result.get("adapters", {})
        ]

        scores = [
            correctness_score(adapter_result)
            for adapter_result in adapter_results
        ]

        likely_correct = sum(
            correctness_verdict(adapter_result) == "likely_correct"
            for adapter_result in adapter_results
        )

        wins = 0
        for result in results:
            available = result.get("adapters", {})
            if not available:
                continue

            best_adapter = max(
                available,
                key=lambda name: correctness_score(available[name]),
            )

            if best_adapter == adapter_name:
                wins += 1

        summary_rows.append({
            "Adapter": adapter_name,
            "Evaluated": len(scores),
            "Average correctness": (
                round(sum(scores) / len(scores), 4)
                if scores else 0
            ),
            "Likely correct": likely_correct,
            "Best-response wins": wins,
        })

    summary = pd.DataFrame(summary_rows).sort_values(
        "Average correctness",
        ascending=False,
    )

    display(Markdown("## Adapter inference summary"))
    display(
        summary.style
        .format({"Average correctness": "{:.3f}"})
        .hide(axis="index")
    )

    # Select either the strongest prompts or a reproducible random sample.
    if selection == "best":
        selected = sorted(
            results,
            key=lambda result: max(
                (
                    correctness_score(adapter_result)
                    for adapter_result
                    in result.get("adapters", {}).values()
                ),
                default=0,
            ),
            reverse=True,
        )[:limit]
    elif selection == "random":
        selected = random.Random(seed).sample(
            results,
            min(limit, len(results)),
        )
    else:
        raise ValueError("selection must be 'best' or 'random'.")

    display(
        Markdown(
            f"## {len(selected)} adapter response comparisons"
        )
    )

    for index, result in enumerate(selected, start=1):
        instruction = result.get("instruction", "")
        reference = result.get("reference_response", "")

        if len(reference) > reference_preview_chars:
            reference = reference[:reference_preview_chars].rstrip() + "…"

        display(Markdown(f"### {index}. {instruction}"))
        display(Markdown(f"**Reference:** {reference}"))

        response_rows = []

        for adapter_name in adapter_names:
            adapter_result = result.get("adapters", {}).get(adapter_name)

            if not adapter_result:
                continue

            response_rows.append({
                "Adapter": adapter_name,
                "Generated response": adapter_result.get("response", ""),
                "Correctness score": correctness_score(adapter_result),
                "Verdict": correctness_verdict(adapter_result),
            })

        response_table = pd.DataFrame(response_rows).sort_values(
            "Correctness score",
            ascending=False,
        )

        display(
            response_table.style
            .format({"Correctness score": "{:.3f}"})
            .set_properties(
                subset=["Generated response"],
                **{
                    "white-space": "pre-wrap",
                    "text-align": "left",
                    "max-width": "700px",
                },
            )
            .hide(axis="index")
        )

    best_adapter = summary.iloc[0]["Adapter"]

    print(
        "\nBest adapter across the complete evaluation set:",
        best_adapter,
    )

    return summary, selected


# results = compare_all_adapters(
#     base_model_folder=CPT_MODEL_DIR,
#     adapters_parent=RUN_OUTPUT_DIR,
#     evaluation_jsonl=INSTRUCTION_DATA_DIR / "evaluation.jsonl",
#     output_json=EVALUATION_OUTPUT_DIR / "adapter_comparison.json",
#     max_new_tokens=200,
#     batch_size=4,
#     sample_size=10,
#     sample_mode="best",       # Change to "random" for a fixed random sample
#     correctness_threshold=0.30,
# )


In [15]:
print_adapter_comparison(RUN_OUTPUT_DIR)


QLoRA ADAPTER CONFIGURATION COMPARISON
Parent folder: /home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3

+======================+===============+========+=========+===========+================================+
| Run                  | Assignment    | Rank   | Alpha   | Dropout   | Target modules                 |
+======================+===============+========+=========+===========+================================+
| adapter-a            | A / Low       | 8      | 16      | 0.05      | q_proj, v_proj                 |
+----------------------+---------------+--------+---------+-----------+--------------------------------+
| adapter-b            | B / Balanced  | 16     | 32      | 0.05      | q_proj, v_proj                 |
+----------------------+---------------+--------+---------+-----------+--------------------------------+
| adapter-c            | C / High      | 32     | 32      | 0.05      | o_proj, q_proj, v_proj         |
+----------------------+---------------+--------+------

[{'directory': 'adapter-a',
  'capacity': 'A / Low',
  'base_model': '/home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model',
  'rank': 8,
  'alpha': 16,
  'dropout': 0.05,
  'targets': 'q_proj, v_proj',
  'initial_loss': 3.538893222808838,
  'final_loss': 3.160775661468506,
  'initial_ppl': 34.42879309328681,
  'final_ppl': 23.588885824586622,
  'ppl_change_percent': -31.485005121523802,
  'train_loss': 3.277614776293437,
  'runtime': 153.2046,
  'epochs': 3.0,
  'best_epoch': 3.0,
  'best_eval_loss': 3.160775661468506},
 {'directory': 'adapter-b',
  'capacity': 'B / Balanced',
  'base_model': '/home/jovyan/llmassign/1a/output/Qwen2.5-1.5B/v3/final_model',
  'rank': 16,
  'alpha': 32,
  'dropout': 0.05,
  'targets': 'q_proj, v_proj',
  'initial_loss': 3.538893222808838,
  'final_loss': 3.0483012199401855,
  'initial_ppl': 34.42879309328681,
  'final_ppl': 21.079504547131386,
  'ppl_change_percent': -38.773617506674576,
  'train_loss': 3.174680463473002,
  'runtime': 155.3505,
 

In [16]:


results = compare_all_adapters(
    base_model_folder=CPT_MODEL_DIR,
    adapters_parent=RUN_OUTPUT_DIR,
    evaluation_jsonl=INSTRUCTION_DATA_DIR / "evaluation.jsonl",
    output_json=EVALUATION_OUTPUT_DIR / "adapter_comparison.json",
    max_new_tokens=200,
    batch_size=4,
    sample_size=10,
    sample_mode="best",       # Change to "random" for a fixed random sample
    correctness_threshold=0.30,
)




Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


OVERALL HELD-OUT EVALUATION


,Adapter,Average F1,Likely correct
1,adapter-b,0.1753,8/40
0,adapter-a,0.1533,1/40
2,adapter-c,0.1531,6/40



BEST 10 RESPONSE COMPARISON


,#,Instruction,adapter-a,adapter-b,adapter-c,Best
0,1,Which Cisco N9000 platforms provide leaf and spine functions in an ACI fabric?,"0.47 | Cisco N9300, N9500, and N9800 platforms provide leaf and spine functions in an ACI fabric.莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿莿","0.41 | Cisco N9300, N9500, and N9800 platforms support leaf and spine functions in an ACI fabric.处置 处置 处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置","0.35 | The N9300, N9500, and N9800 platforms support leaf and spine functions in an ACI fabric. الجزائ الجزائ الجزائ الجزائ الجزائ الجزائ الج",adapter-a
1,2,What information is exchanged in OSPF Hello packets?,0.14 | OSPF Hello packets are used to discover and establish neighbor relationships in the OSPF routing domain.إصد إصدrouter What information is ex,0.41 | Hello packets are used to discover and establish neighbor relationships in the OSPF routing domain.处置 处置 处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处,"0.42 | Hello packets are used to discover and establish neighbor relationships, and to exchange link state advertisements (LSAs) with neighbors. ال",adapter-c
2,3,What is the purpose of an OSPF stub area?,0.11 | An OSPF stub area is a type of OSPF area that has limited reachability to other areas. It is used to reduce the amount of routing informatio,"0.31 | A stub area is a type of OSPF area that is configured to prevent the flooding of certain types of LSAs, such as AS-external-LSAs, to reduce",0.40 | A stub area is a type of OSPF area that is configured to prevent external routing and limit the propagation of routing information.核心区 核心区 核,adapter-c
3,4,What is residual bandwidth in OSPF performance-metric extensions?,0.15 | Residual bandwidth is a performance metric used in OSPF (Open Shortest Path First) to measure the remaining capacity of a link after account,0.36 | Residual bandwidth is the amount of bandwidth that is available for use by a router or a network segment.处置 处置 处置处置处置处置处置处置处置处置处置处置处置处置处置处置处,0.11 | It is the amount of bandwidth remaining after accounting for the use of the link for traffic that is not being forwarded by the protocol. ال,adapter-b
4,5,How should an OSPF area border router handle entropy-label capability between areas?,0.24 | An OSPF area border router should advertise the OSPF area border router capability in the OSPFv2 Router Information LSA.处置 处置处置处置处置处置处置处置处置处,0.35 | The OSPF area border router should advertise the capability in the OSPF Hello packets it sends to the area.处置 处置 处置处置处置处置处置处置处置处置处置处置处置处置处置处,"0.20 | It should advertise the capability to all areas that it can reach, and it should not advertise the capability to areas that it cannot reach.",adapter-b
5,6,What does an available-bandwidth advertisement tell a traffic-engineering controller?,0.20 | An available-bandwidth advertisement tells a traffic-engineering controller the amount of bandwidth that is available for a particular path.,0.35 | It tells the controller how much bandwidth is available on a link.处置 处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处置处,"0.06 | It tells the controller the maximum bandwidth that a link can provide, and it can be used to set up traffic policies that ensure that traffi",adapter-b
6,7,How can OSPF use equal-cost paths?,"0.17 | Equal-cost multipath (ECMP) is a feature that allows multiple paths to a destination to be used simultaneously, with the same cost, to impro","0.11 | Equal-cost multipath (ECMP) is a feature that allows multiple paths to a destination to be used simultaneously, with the same cost, to reduc","0.33 | It can use multiple paths to the same destination, with the same metric, and distribute the load evenly. الجزائ الجزائ الجزائ الجزائ الج",adapter-c
7,8,What is an OSPF area border router?,0.28 | An OSPF area border router is a router that connects to multiple areas in an OSPF network. It is responsible for advertising the topology of,0.33 | An OSPF area border router is a router that connects to multiple areas and has the capability 


Full responses saved to: /home/jovyan/llmassign/1a/evaluation/Qwen2.5-1.5B/v3/adapter_comparison.json


<a id="final-results-and-recommendation"></a>
## Final Results and Recommendation

All three adapters improved held-out response prediction relative to the untrained adapter state. Increasing capacity progressively reduced validation perplexity, but generated-answer quality did not follow the same ordering.

| Adapter | Final validation PPL | PPL reduction | Average response F1 | Likely correct | Runtime | Overall interpretation |
|---|---:|---:|---:|---:|---:|---|
| A | 23.589 | 31.49% | 0.1533 | 1/40 | 153.2 s | Lowest-capacity baseline; it improved validation PPL but produced the weakest correctness count. |
| **B** | 21.080 | 38.77% | **0.1753** | **8/40** | 155.4 s | **Best generated-response result and best quality/cost balance.** |
| C | **18.200** | **47.14%** | 0.1531 | 6/40 | 169.0 s | Best teacher-forced loss and PPL, but its additional capacity did not produce the best generated-answer score. |

### Final verdict

**Adapter B is selected as the final instruction-tuned adapter.** Adapter C achieved the lowest held-out loss and perplexity, reducing PPL from 34.429 to 18.200, which shows that it predicts the fixed evaluation responses most confidently under teacher forcing. However, Adapter B achieved the highest average token F1 and the largest number of responses above the stated correctness threshold: 8 of 40, compared with 6 for Adapter C and 1 for Adapter A. Its 155.4-second runtime was also close to Adapter A and lower than Adapter C.

The automatic F1 score measures lexical overlap with grounded reference responses and is not a substitute for expert factual review. The relatively low absolute scores also show that generated-answer quality still has room for improvement. Nevertheless, among the three controlled runs, the aggregate evaluation and side-by-side responses support Adapter B as the best quality-and-cost trade-off for the Cisco data-center assistant.